In this notebook, we explore our data sample with previous data from the literature:

- Cao et al. (2015): full sample
- Cao et al. (2017): same as Cao et al. (2015), only filters
- Chen et al. (2019): full sample
- Liu et al. (2022): same as Chen et al. (2019), only filters

In [1]:
import numpy as np
import pandas as pd
from astropy import units as u

database = pd.read_csv(
    "/home/renan/slcomp/02_Data/02_Database/Database.csv", low_memory=False
)

---

In [2]:
data_sample = pd.read_csv("01_LaStBeRu_cosmo_ground.csv").convert_dtypes()
data_sample.head()

,JNAME,RA,DEC,System_Type,Lens_Type,Source_Type,theta_E,theta_E_rad,theta_EErr,theta_EMethod,...,petror50_r,devrad_r_rad,exprad_r_rad,petror50_r_rad,velDisp0_dev,velDisp0Err_dev,velDisp0_exp,velDisp0Err_exp,velDisp0_petro,velDisp0Err_petro
0,J000408.1+141150.7,1.03358,14.19742,<NA>,<NA>,<NA>,0.57,0.000003,<NA>,<NA>,...,0.855237,0.000003,0.000003,0.000004,188.674241,40.769163,189.882588,41.15103,183.653023,39.283495
1,J001016.4+175852.2,2.56854,17.98116,<NA>,<NA>,<NA>,1.23,0.000006,<NA>,<NA>,...,1.770891,0.000009,0.000005,0.000009,263.17974,41.536244,273.211287,43.552805,263.135764,41.528739
2,J001321.4+114039.6,3.33925,11.67767,<NA>,<NA>,<NA>,1.07,0.000005,<NA>,<NA>,...,1.260727,0.000005,0.000003,0.000006,233.373159,17.523127,239.686481,19.200784,229.993206,16.84232
3,J002123.2-042926.2,5.34679,-4.4906,<NA>,<NA>,<NA>,0.95,0.000005,<NA>,<NA>,...,1.524729,0.000009,0.000005,0.000007,232.211211,34.322592,241.619291,36.074245,235.816734,34.918168
4,J002240.9+143110.1,5.67046,14.51948,Single Lens Galaxy,GALAXY,GALAXY,2.91,0.000014,0.14,SIE model,...,2.548123,0.000018,0.000008,0.000012,391.881959,56.013125,413.299356,59.597044,401.293768,57.340201


In [3]:
data_sample.keys()

Index(['JNAME', 'RA', 'DEC', 'System_Type', 'Lens_Type', 'Source_Type',
       'theta_E', 'theta_E_rad', 'theta_EErr', 'theta_EMethod', 'theta_ERef',
       'z_L', 'z_LErr', 'z_LType', 'z_LRef', 'z_S', 'z_SErr', 'z_SType',
       'z_SRef', 'velDisp', 'velDispErr', 'velDispRef', 'mag_u', 'mag_uErr',
       'mag_uRef', 'mag_uS', 'mag_uSRef', 'mag_g', 'mag_gErr', 'mag_gRef',
       'mag_gS', 'mag_gSRef', 'mag_r', 'mag_rErr', 'mag_rRef', 'mag_rS',
       'mag_rSRef', 'mag_i', 'mag_iErr', 'mag_iRef', 'mag_iS', 'mag_iSRef',
       'mag_z', 'mag_zErr', 'mag_zRef', 'mag_zS', 'mag_zSRef', 'mag_F814W',
       'mag_F814WErr', 'mag_F814WRef', 'fiberid', 'mjd', 'plate', 'theta_ap',
       'theta_ap_rad', 'seeing20', 'seeing20_rad', 'seeing50', 'seeing50_rad',
       'seeing80', 'seeing80_rad', 'devrad_r', 'exprad_r', 'petror50_r',
       'devrad_r_rad', 'exprad_r_rad', 'petror50_r_rad', 'velDisp0_dev',
       'velDisp0Err_dev', 'velDisp0_exp', 'velDisp0Err_exp', 'velDisp0_petro',
       'velDisp0Er

# 1. Data from Cao et al. (2015)

In [4]:
data_cao = pd.read_csv("data/cao_et_al.(2015).csv").convert_dtypes()
data_cao.head()

,Name,z_l,z_s,sigma_ap,sigma_apErr,theta_E,Survey,theta_ap,theta_eff,sigma0,sigma0Err
0,J0151+0049,0.517,1.364,219,39,0.68,BELLS,1.0,0.89,226,40
1,J0747+5055,0.438,0.898,328,60,0.75,BELLS,1.0,1.24,334,61
2,J0747+4448,0.437,0.897,281,52,0.61,BELLS,1.0,2.87,277,51
3,J0801+4727,0.483,1.518,98,24,0.49,BELLS,1.0,0.57,103,25
4,J0830+5116,0.53,1.332,268,36,1.14,BELLS,1.0,1.1,274,37


In [5]:
data_cao = data_cao.rename(
    columns={
        "Name": "Cao_Name",
        "z_l": "Cao_z_L",
        "z_s": "Cao_z_S",
        "sigma_ap": "Cao_velDisp",
        "sigma_apErr": "Cao_velDispErr",
        "theta_E": "Cao_theta_E",
        "Survey": "Cao_Survey",
        "theta_ap": "Cao_theta_ap",
        "theta_eff": "Cao_theta_eff",
        "sigma0": "Cao_velDisp0",
        "sigma0Err": "Cao_velDisp0Err",
    }
)

In this paper, they already evaluate $\sigma_0$ but using different parameters than Chen et al. (2019). Thus, we re-evaluate $\sigma_0$ using the same quantities we will apply in the next steps:

In [6]:
@np.vectorize
def cal_velDispErr(velDisp, velDispErr, theta_eff, theta_ap, velDisp0):
    term_1 = (velDispErr**2) / (velDisp**2)
    term_2 = 0.03**2
    term_3 = (np.log(theta_eff / (2 * theta_ap)) * 0.033) ** 2
    return np.sqrt((term_1 + term_2 + term_3) * velDisp0**2)

In [7]:
data_cao["Cao_velDisp0"] = data_cao["Cao_velDisp"] * (
    (data_cao["Cao_theta_eff"] / (2 * data_cao["Cao_theta_ap"])) ** (-0.066)
)
data_cao["Cao_velDisp0Err"] = cal_velDispErr(
    data_cao["Cao_velDisp"],
    data_cao["Cao_velDispErr"],
    data_cao["Cao_theta_eff"],
    data_cao["Cao_theta_ap"],
    data_cao["Cao_velDisp0"],
)

In [8]:
# for j in data_cao['Cao_Name']:
#     print(j, database.query(f'Original_ID.astype("str").str.contains("{j}",regex=False) or Alternative_Name.astype("str").str.contains("{j}",regex=False)').JNAME.values)
dictionary_cao_jname = {
    "J0151+0049": "J015107.4+004909.0",
    "J0747+5055": "J074724.1+505537.5",
    "J0747+4448": "J074734.8+444859.3",
    "J0801+4727": "J080105.3+472749.6",
    "J0830+5116": "J083049.7+511631.8",
    "J0944-0147": "J094427.5-014742.4",
    "J1159-0007": "J115944.6-000728.2",
    "J1215+0047": "J121504.4+004726.0",
    "J1221+3806": "J122151.9+380610.5",
    "J1234-0241": "J123428.0-024129.6",
    "J1318-0104": "J131829.4-010421.6",
    "J1337+3620": "J133751.3+362018.1",
    "J1349+3612": "J134910.3+361239.7",
    "J1352+3216": "J135219.0+321651.8",
    "J1522+2910": "J152209.5+291021.9",
    "J1541+1812": "J154118.6+181235.1",
    "J1542+1629": "J154246.3+162951.8",
    "J1545+2748": "J154503.6+274805.3",
    "J1601+2138": "J160113.3+213833.9",
    "J1611+1705": "J161109.8+170526.6",
    "J1631+1854": "J163150.3+185404.1",
    "J1637+1439": "J163714.6+143930.1",
    "J2122+0409": "J212252.0+040935.5",
    "J2125+0411": "J212510.7+041131.6",
    "J2303+0037": "J230335.2+003703.2",
    "J0008-0004": "J000803.0-000408.2",
    "J0029-0055": "J002907.8-005550.5",
    "J0037-0942": "J003753.1-094218.3",
    "J0044+0113": "J004402.9+011312.6",
    "J0109+1500": "J010933.7+150032.5",
    "J0157-0056": "J015758.9-005626.1",
    "J0216-0813": "J021652.5-081345.3",
    "J0252+0039": "J025245.2+003958.4",
    "J0330-0020": "J033012.1-002051.9",
    "J0405-0455": "J040535.4-045552.4",
    "J0728+3835": "J072805.0+383525.7",
    "J0737+3216": "J073728.5+321618.6",
    "J0808+4706": "J080858.8+470638.9",
    "J0822+2652": "J082242.3+265243.5",
    "J0841+3824": "J084128.8+382413.7",
    "J0903+4116": "J090315.2+411609.1",
    "J0912+0029": "J091205.3+002901.2",
    "J0935-0003": "J093543.9-000334.8",
    "J0936+0913": "J093600.8+091335.8",
    "J0946+1006": "J094656.7+100652.8",
    "J0956+5100": "J095629.8+510006.6",
    "J0959+0410": "J095944.1+041017.0",
    "J1016+3859": "J101622.9+385903.3",
    "J1020+1122": "J102026.5+112241.1",
    "J1023+4230": "J102332.3+423001.8",
    "J1100+5329": "J110024.4+532913.9",
    "J1106+5228": "J110646.1+522837.8",
    "J1112+0826": "J111250.6+082610.4",
    "J1134+6027": "J113405.9+602713.5",
    "J1142+1001": "J114257.3+100111.8",
    "J1143-0144": "J114329.6-014430.0",
    "J1153+4612": "J115310.8+461205.3",
    "J1204+0358": "J120444.1+035806.4",
    "J1205+4910": "J120540.4+491029.4",
    "J1213+6708": "J121340.6+670829.0",
    "J1218+0830": "J121826.7+083050.3",
    "J1250+0523": "J125028.3+052349.1",
    "J1251-0208": "J125135.7-020805.2",
    "J1330-0148": "J133045.5-014841.6",
    "J1402+6321": "J140228.2+632133.5",
    "J1403+0006": "J140329.5+000641.4",
    "J1416+5136": "J141622.3+513630.4",
    "J1430+4105": "J143004.1+410557.2",
    "J1436-0000": "J143627.5-000029.2",
    "J1451-0239": "J145128.2-023936.4",
    "J1525+3327": "J152506.7+332747.4",
    "J1531-0105": "J153150.1-010545.7",
    "J1538+5817": "J153812.9+581709.8",
    "J1621+3931": "J162133.0+393144.6",
    "J1627-0053": "J162746.5-005357.6",
    "J1630+4520": "J163028.2+452036.3",
    "J1636+4707": "J163602.6+470729.6",
    "J2238-0754": "J223840.1-075456.0",
    "J2300+0022": "J230053.2+002238.0",
    "J2303+1422": "J230321.7+142217.9",
    "J2321-0939": "J232120.9-093910.3",
    "J2341+0000": "J234111.6+000018.7",
    "Q0047-2808": "J004941.9-275226.0",
    "CFRS03-1077": "J030230.7+000604.6",
    "HST 14176": "J141736.1+522642.9",
    "HST 15433": "J154320.4+535152.0",
    "MG 2016": "J201918.1+112710.8",
    "J0212-0555": "J021247.0-055552.0",
    "J0213-0743": "J021324.8-074354.9",
    "J0214-0405": "J021411.1-040502.4",
    "J0217-0513": "J021737.1-051329.2",
    "J0219-0829": "J021902.1-082934.4",
    "J0223-0534": "J022346.1-053418.2",
    "J0225-0454": "J022511.0-045433.4",
    "J0226-0420": "J022610.3-042011.3",
    "J0232-0408": "J023251.3-040823.4",
    "J0848-0351": "J084847.1-035103.3",
    "J0849-0412": "J084909.3-041226.4",
    "J0849-0251": "J084959.0-025142.0",
    "J0850-0347": "J085019.0-034710.4",
    "J0855-0147": "J085540.1-014730.2",
    "J0855-0409": "J085559.0-040917.0",
    "J0904-0059": "J090407.0-005952.0",
    "J0959+0206": "J095921.0+020638.0",
    "J1359+5535": "J135949.3+553550.2",
    "J1404+5200": "J140454.2+520024.4",
    "J1405+5243": "J140546.1+524311.1",
    "J1406+5226": "J140650.2+522619.1",
    "J1411+5651": "J141137.0+565119.0",
    "J1420+5258": "J142031.4+525822.0",
    "J1420+5630": "J142059.4+563007.2",
    "J2203+0205": "J220329.0+020518.6",
    "J2205+0147": "J220506.0+014703.0",
    "J2213-0009": "J221326.1-000946.2",
    "J2219-0017": "J221929.2-001743.2",
    "J2220+0106": "J222012.0+010606.0",
    "J2221+0115": "J222148.0+011542.0",
    "J2222+0012": "J222217.3+001202.4",
}

In [9]:
data_cao["JNAME"] = data_cao["Cao_Name"].apply(lambda x: dictionary_cao_jname[x])

Converting other variables to radians to use it later:

In [10]:
data_cao["Cao_theta_E_rad"] = data_cao["Cao_theta_E"] * u.arcsec.to(u.rad)
data_cao["Cao_theta_ap_rad"] = data_cao["Cao_theta_ap"] * u.arcsec.to(u.rad)
data_cao["Cao_theta_eff_rad"] = data_cao["Cao_theta_eff"] * u.arcsec.to(u.rad)

In [11]:
sum(data_cao.JNAME.isin(data_sample.JNAME))

54

In [12]:
data_cao["in_lastberu_cosmo_ground"] = data_cao.JNAME.isin(data_sample.JNAME)

In [13]:
sum(data_cao["in_lastberu_cosmo_ground"])

54

# 2. Cao et al. (2017)

In this work, they applied the following cuts on previous data: 
$$ 200\leq\sigma_{\mathrm{ap}}\leq300$$

In [14]:
data_cao.query("Cao_velDisp>=200 and Cao_velDisp<=300", engine="python").shape

(80, 16)

In [15]:
data_cao["in_cao_2017"] = (data_cao["Cao_velDisp"] >= 200) & (
    data_cao["Cao_velDisp"] <= 300
)

In [16]:
data_cao.keys()

Index(['Cao_Name', 'Cao_z_L', 'Cao_z_S', 'Cao_velDisp', 'Cao_velDispErr',
       'Cao_theta_E', 'Cao_Survey', 'Cao_theta_ap', 'Cao_theta_eff',
       'Cao_velDisp0', 'Cao_velDisp0Err', 'JNAME', 'Cao_theta_E_rad',
       'Cao_theta_ap_rad', 'Cao_theta_eff_rad', 'in_lastberu_cosmo_ground',
       'in_cao_2017'],
      dtype='object')

In [17]:
data_cao[
    [
        "JNAME",
        "Cao_Name",
        "in_cao_2017",
        "in_lastberu_cosmo_ground",
        "Cao_z_L",
        "Cao_z_S",
        "Cao_velDisp",
        "Cao_velDispErr",
        "Cao_velDisp0",
        "Cao_velDisp0Err",
        "Cao_theta_E",
        "Cao_theta_E_rad",
        "Cao_theta_ap",
        "Cao_theta_ap_rad",
        "Cao_theta_eff",
        "Cao_theta_eff_rad",
        "Cao_Survey",
    ]
].to_csv("03_Cao_et_al._(2015)_Data.csv", index=False)

# 3. Chen et al. (2019)

In [18]:
data_chen = pd.read_csv("data/chen_et_al.(2019).csv").convert_dtypes()
data_chen.head()

,Lens name,zl,zs,theta_E,theta_Eff,slit,Fiber radius,theta_ap,sigma_ap,sigma_apErr,Survey name
0,MG2016+112,1.004,3.263,1.56,0.31,1 × 1.25,<NA>,0.65,304,27,LSD
1,0047−281,0.485,3.595,1.34,0.82,0.4 × 1.25,<NA>,0.41,219,12,LSD
2,CFRS03.1077,0.938,2.941,1.24,1.6,0.5 × 1.25,<NA>,0.46,256,19,LSD
3,HST14176+5226,0.81,3.399,1.41,1.06,0.32 × 1.25,<NA>,0.37,212,18,LSD
4,HSTT15433+5352,0.497,2.092,0.36,0.41,0.3 × 1.25,<NA>,0.35,108,14,LSD


In [19]:
data_chen = data_chen.rename(
    columns={
        "Lens name": "Chen_Name",
        "zl": "Chen_z_L",
        "zs": "Chen_z_S",
        "theta_E": "Chen_theta_E",
        "theta_Eff": "Chen_theta_eff",
        "slit": "Chen_slit",
        "Fiber radius": "Chen_fiber_radius",
        "theta_ap": "Chen_theta_ap",
        "sigma_ap": "Chen_velDisp",
        "sigma_apErr": "Chen_velDispErr",
        "Survey name": "Chen_Survey",
    }
)

In [20]:
data_chen["Chen_velDisp0"] = data_chen["Chen_velDisp"] * (
    (data_chen["Chen_theta_eff"] / (2 * data_chen["Chen_theta_ap"])) ** (-0.066)
)
data_chen["Chen_velDisp0Err"] = cal_velDispErr(
    data_chen["Chen_velDisp"],
    data_chen["Chen_velDispErr"],
    data_chen["Chen_theta_eff"],
    data_chen["Chen_theta_ap"],
    data_chen["Chen_velDisp0"],
)

In [21]:
# for j in data_chen['Chen_Name']:
#     print(j, database.query(f'Original_ID.astype("str").str.contains("{j}",regex=False) or Alternative_Name.astype("str").str.contains("{j}",regex=False)').JNAME.values)
dictionary_chen_jname = {
    "MG2016+112": "J201918.1+112710.8",
    "0047−281": "J004941.9-275226.0",
    "CFRS03.1077": "J030230.7+000604.6",
    "HST14176+5226": "J141736.1+522642.9",
    "HSTT15433+5352": "J154320.4+535152.0",
    "SL2SJ020524−93023": "",
    "SL2SJ021247−055552": "J021247.0-055552.0",
    "SL2SJ021325−074355": "J021324.8-074354.9",
    "SL2SJ021411−040502": "J021411.1-040502.4",
    "SL2SJ021737−051329": "J021737.1-051329.2",
    "SL2SJ021801−080247": "J021801.1-080247.3",
    "SL2SJ021902−082934": "J021902.1-082934.4",
    "SL2SJ022046−094927": "J022046.0-094927.4",
    "SL2SJ022511−045433": "J022511.0-045433.4",
    "SL2SJ022610−042011": "J022610.3-042011.3",
    "SL2SJ023251−040823": "J023251.3-040823.4",
    "SL2SJ084847−035103": "J084847.1-035103.3",
    "SL2SJ084909−041226": "J084909.3-041226.4",
    "SL2SJ084959−025142": "J084959.0-025142.0",
    "SL2SJ085540−014730": "J085540.1-014730.2",
    "SL2SJ090407−005952": "J090407.0-005952.0",
    "SL2SJ095921+020638": "J095921.0+020638.0",
    "SL2SJ135949+553550": "J135949.3+553550.2",
    "SL2SJ140454+520024": "J140454.2+520024.4",
    "SL2SJ140546+524311": "J140546.1+524311.1",
    "SL2SJ140650+522619": "J140650.2+522619.1",
    "SL2SJ141137+565119": "J141137.0+565119.0",
    "SL2SJ142059+563007": "J142059.4+563007.2",
    "SL2SJ220329+020518": "J220329.0+020518.6",
    "SL2SJ220506+014703": "J220506.0+014703.0",
    "SL2SJ222148+011542": "J222148.0+011542.0",
    "SDSSJ0008−0004": "J000803.0-000408.2",
    "SDSSJ0029−0055": "J002907.8-005550.5",
    "SDSSJ0037−0942": "J003753.1-094218.3",
    "SDSSJ0044+0113": "J004402.9+011312.6",
    "SDSSJ0109+1500": "J010933.7+150032.5",
    "SDSSJ0157−0056": "J015758.9-005626.1",
    "SDSSJ0216−0813": "J021652.5-081345.3",
    "SDSSJ0252+0039": "J025245.2+003958.4",
    "SDSSJ0330−0020": "J033012.1-002051.9",
    "SDSSJ0405−0455": "J040535.4-045552.4",
    "SDSSJ0728+3835": "J072805.0+383525.7",
    "SDSSJ0737+3216": "J073728.5+321618.6",
    "SDSSJ0822+2652": "J082242.3+265243.5",
    "SDSSJ0903+4116": "J090315.2+411609.1",
    "SDSSJ0912+0029": "J091205.3+002901.2",
    "SDSSJ0935−0003": "J093543.9-000334.8",
    "SDSSJ0936+0913": "J093600.8+091335.8",
    "SDSSJ0946+1006": "J094656.7+100652.8",
    "SDSSJ0956+5100": "J095629.8+510006.6",
    "SDSSJ0959+4416": "J095901.0+441639.4",
    "SDSSJ0959+0410": "J095944.1+041017.0",
    "SDSSJ1016+3859": "J101622.9+385903.3",
    "SDSSJ1020+1122": "J102026.5+112241.1",
    "SDSSJ1023+4230": "J102332.3+423001.8",
    "SDSSJ1029+0420": "J102922.9+042001.8",
    "SDSSJ1100+5329": "J110024.4+532913.9",
    "SDSSJ1106+5228": "J110646.1+522837.8",
    "SDSSJ1112+0826": "J111250.6+082610.4",
    "SDSSJ1134+6027": "J113405.9+602713.5",
    "SDSSJ1142+1001": "J114257.3+100111.8",
    "SDSSJ1143−0144": "J114329.6-014430.0",
    "SDSSJ1153+4612": "J115310.8+461205.3",
    "SDSSJ1204+0358": "J120444.1+035806.4",
    "SDSSJ1205+4910": "J120540.4+491029.4",
    "SDSSJ1213+6708": "J121340.6+670829.0",
    "SDSSJ1218+0830": "J121826.7+083050.3",
    "SDSSJ1250+0523": "J125028.3+052349.1",
    "SDSSJ1402+6321": "J140228.2+632133.5",
    "SDSSJ1403+0006": "J140329.5+000641.4",
    "SDSSJ1416+5136": "J141622.3+513630.4",
    "SDSSJ1420+6019": "J142015.8+601914.8",
    "SDSSJ1430+4105": "J143004.1+410557.2",
    "SDSSJ1436−0000": "J143627.5-000029.2",
    "SDSSJ1443+0304": "J144319.6+030408.2",
    "SDSSJ1451−0239": "J145128.2-023936.4",
    "SDSSJ1525+3327": "J152506.7+332747.4",
    "SDSSJ1531−0105": "J153150.1-010545.7",
    "SDSSJ1538+5817": "J153812.9+581709.8",
    "SDSSJ1621+3931": "J162133.0+393144.6",
    "SDSSJ1627−0053": "J162746.5-005357.6",
    "SDSSJ1630+4520": "J163028.2+452036.3",
    "SDSSJ1636+4707": "J163602.6+470729.6",
    "SDSSJ2238−0754": "J223840.1-075456.0",
    "SDSSJ2300+0022": "J230053.2+002238.0",
    "SDSSJ2303+1422": "J230321.7+142217.9",
    "SDSSJ2321−0939": "J232120.9-093910.3",
    "SDSSJ2341+0000": "J234111.6+000018.7",
    "SDSSJ0143−1006": "J014356.6-100633.7",
    "SDSSJ0159−0006": "J015930.1-000612.4",
    "SDSSJ0324+0045": "J032415.5+004505.5",
    "SDSSJ0324−0110": "J032454.5-011029.1",
    "SDSSJ0753+3416": "J075346.2+341633.6",
    "SDSSJ0754+1927": "J075428.5+192728.1",
    "SDSSJ0757+1956": "J075749.0+195616.3",
    "SDSSJ0826+5630": "J082639.9+563036.0",
    "SDSSJ0847+2348": "J084727.7+234819.5",
    "SDSSJ0851+0505": "J085141.9+050507.0",
    "SDSSJ0920+3028": "J092048.3+302818.3",
    "SDSSJ0955+3014": "J095557.5+301450.9",
    "SDSSJ0956+5539": "J095654.8+553947.3",
    "SDSSJ1010+3124": "J101026.8+312417.6",
    "SDSSJ1041+0112": "J104122.9+011224.2",
    "SDSSJ1048+1313": "J104809.4+131352.9",
    "SDSSJ1051+4439": "J105109.4+443908.5",
    "SDSSJ1056+4141": "J105657.6+414114.6",
    "SDSSJ1101+1523": "J110113.1+152339.6",
    "SDSSJ1116+0729": "J111641.7+072945.6",
    "SDSSJ1127+2312": "J112738.7+231244.4",
    "SDSSJ1137+1818": "J113728.6+181812.4",
    "SDSSJ1142+2509": "J114238.2+250905.5",
    "SDSSJ1144+0436": "J114440.1+043650.5",
    "SDSSJ1213+2930": "J121303.7+293022.4",
    "SDSSJ1301+0834": "J130126.9+083425.2",
    "SDSSJ1330+1750": "J133031.4+175040.5",
    "SDSSJ1403+3309": "J140309.7+330917.8",
    "SDSSJ1430+6104": "J143034.8+610404.8",
    "SDSSJ1433+2835": "J143351.6+283516.4",
    "SDSSJ1541+3642": "J154122.3+364231.7",
    "SDSSJ1543+2202": "J154339.9+220223.3",
    "SDSSJ1550+2020": "J155010.6+202013.5",
    "SDSSJ1553+3004": "J155316.1+300425.7",
    "SDSSJ1607+2147": "J160740.5+214711.0",
    "SDSSJ1633+1441": "J163344.2+144154.9",
    "SDSSJ2309−0039": "J230946.4-003912.9",
    "SDSSJ2324+0105": "J232427.8+010558.5",
    "SDSSJ0801+4727": "J080105.3+472749.6",
    "SDSSJ1234−0241": "J123428.0-024129.6",
    "SDSSJ1352+3216": "J135219.0+321651.8",
    "SDSSJ1159−0007": "J115944.6-000728.2",
    "SDSSJ1318−0104": "J131829.4-010421.6",
    "SDSSJ1349+3612": "J134910.3+361239.7",
    "SDSSJ1221+3806": "J122151.9+380610.5",
    "SDSSJ0944−0147": "J094427.5-014742.4",
    "SDSSJ1601+2138": "J160113.3+213833.9",
    "SDSSJ1542+1629": "J154246.3+162951.8",
    "SDSSJ0151+0049": "J015107.4+004909.0",
    "SDSSJ1337+3620": "J133751.3+362018.1",
    "SDSSJ2125+0411": "J212510.7+041131.6",
    "SDSSJ1545+2748": "J154503.6+274805.3",
    "SDSSJ1215+0047": "J121504.4+004726.0",
    "SDSSJ0830+5116": "J083049.7+511631.8",
    "SDSSJ1631+1854": "J163150.3+185404.1",
    "SDSSJ2303+0037": "J230335.2+003703.2",
    "SDSSJ0747+4448": "J074734.8+444859.3",
    "SDSSJ2122+0409": "J212252.0+040935.5",
    "SDSSJ0747+5055": "J074724.1+505537.5",
    "SDSSJ0029+2544": "J002927.4+254401.8",
    "SDSSJ0201+3228": "J020121.4+322829.7",
    "SDSSJ0237−0641": "J023740.6-064113.0",
    "SDSSJ0742+3341": "J074249.7+334149.0",
    "SDSSJ0755+3445": "J075523.5+344539.6",
    "SDSSJ0856+2010": "J085621.6+201040.5",
    "SDSSJ0918+5104": "J091859.2+510452.6",
    "SDSSJ1110+2808": "J111027.1+280838.5",
    "SDSSJ1116+0915": "J111634.6+091503.1",
    "SDSSJ1141+2216": "J114154.7+221628.9",
    "SDSSJ1201+4743": "J120159.0+474323.2",
    "SDSSJ1226+5457": "J122656.5+545739.1",
    "SDSSJ2228+1205": "J222825.8+120504.0",
    "SDSSJ2342−0120": "J234248.7-012032.6",
}

In [22]:
data_chen["JNAME"] = data_chen["Chen_Name"].apply(lambda x: dictionary_chen_jname[x])

Converting other variables to radians to use it later:

In [23]:
data_chen["Chen_theta_E_rad"] = data_chen["Chen_theta_E"] * u.arcsec.to(u.rad)
data_chen["Chen_theta_ap_rad"] = data_chen["Chen_theta_ap"] * u.arcsec.to(u.rad)
data_chen["Chen_theta_eff_rad"] = data_chen["Chen_theta_eff"] * u.arcsec.to(u.rad)

In [24]:
sum(data_chen.JNAME.isin(data_sample.JNAME))

99

In [25]:
data_chen["in_lastberu_cosmo_ground"] = data_chen.JNAME.isin(data_sample.JNAME)

In [26]:
sum(data_chen["in_lastberu_cosmo_ground"])

99

# 4. Liu et al. (2022)

In this work, they applied the following cuts on previous data:
- 120 SGL data: from surveys SLACS, S4TM, BELLS, and BELLS GALLERY

In [27]:
# data_chen.query('Chen_Survey.isin(["SLACS", "S4TM", "BELLS", "BELLS GALLERY"]) and Chen_z_S < 2.3')
data_chen["in_liu_2022"] = (
    data_chen["Chen_Survey"].isin(["SLACS", "S4TM", "BELLS", "BELLS GALLERY"])
) & (data_chen["Chen_z_S"] <= 2.3)

In [28]:
data_chen.keys()

Index(['Chen_Name', 'Chen_z_L', 'Chen_z_S', 'Chen_theta_E', 'Chen_theta_eff',
       'Chen_slit', 'Chen_fiber_radius', 'Chen_theta_ap', 'Chen_velDisp',
       'Chen_velDispErr', 'Chen_Survey', 'Chen_velDisp0', 'Chen_velDisp0Err',
       'JNAME', 'Chen_theta_E_rad', 'Chen_theta_ap_rad', 'Chen_theta_eff_rad',
       'in_lastberu_cosmo_ground', 'in_liu_2022'],
      dtype='object')

In [29]:
data_chen[
    [
        "JNAME",
        "Chen_Name",
        "in_liu_2022",
        "in_lastberu_cosmo_ground",
        "Chen_z_L",
        "Chen_z_S",
        "Chen_velDisp",
        "Chen_velDispErr",
        "Chen_velDisp0",
        "Chen_velDisp0Err",
        "Chen_theta_E",
        "Chen_theta_E_rad",
        "Chen_theta_ap",
        "Chen_theta_ap_rad",
        "Chen_theta_eff",
        "Chen_theta_eff_rad",
        "Chen_slit",
        "Chen_fiber_radius",
        "Chen_Survey",
    ]
].to_csv("03_Chen_et_al._(2019)_Data.csv", index=False)

Let's compare objects from `Cao et al. (2017)` and the ones in `Liu et al. (2017)`, just to make sure one is contained in the other:

In [30]:
data_cao.query("~JNAME.isin(@data_chen.JNAME)")

,Cao_Name,Cao_z_L,Cao_z_S,Cao_velDisp,Cao_velDispErr,Cao_theta_E,Cao_Survey,Cao_theta_ap,Cao_theta_eff,Cao_velDisp0,Cao_velDisp0Err,JNAME,Cao_theta_E_rad,Cao_theta_ap_rad,Cao_theta_eff_rad,in_lastberu_cosmo_ground,in_cao_2017
14,J1522+2910,0.555,1.311,166,27,0.74,BELLS,1.0,1.08,172.89009,28.810304,J152209.5+291021.9,0.000004,0.000005,0.000005,False,False
15,J1541+1812,0.56,1.113,174,24,0.64,BELLS,1.0,0.59,188.599699,27.684940,J154118.6+181235.1,0.000003,0.000005,0.000003,True,False
19,J1611+1705,0.477,1.211,109,23,0.58,BELLS,1.0,1.33,111.974793,23.912878,J161109.8+170526.6,0.000003,0.000005,0.000006,True,False
21,J1637+1439,0.391,0.874,208,30,0.65,BELLS,1.0,0.89,219.417658,32.851480,J163714.6+143930.1,0.000003,0.000005,0.000004,True,True
37,J0808+4706,0.219,1.025,236,11,1.23,SLACS,1.5,2.42,239.37026,13.376438,J080858.8+470638.9,0.000006,0.000007,0.000012,False,True
39,J0841+3824,0.116,0.657,225,11,1.41,SLACS,1.5,4.21,220.023922,12.858063,J084128.8+382413.7,0.000007,0.000007,0.00002,False,True
62,J1251-0208,0.224,0.784,233,23,0.84,SLACS,1.5,2.61,235.151444,24.284743,J125135.7-020805.2,0.000004,0.000007,0.000013,False,True
63,J1330-0148,0.081,0.712,185,9,0.87,SLACS,1.5,0.89,200.448122,13.995079,J133045.5-014841.6,0.000004,0.000007,0.000004,True,False
92,J0223-0534,0.499,1.44,288,28,1.22,SL2S,1.0,1.31,296.156018,30.414985,J022346.1-053418.2,0.000006,0.000005,0.000006,False,True
99,J0850-0347,0.337,3.25,290,24,0.93,SL2S,0.7,0.28,322.500222,33.156121,J085019.0-034710.4,0.000005,0.000003,0.000001,False,True


There are some systems from Cao et al. (2015) that are not in Chen et al. (2019) sample!

---

# 5. Seeing

For these data, the only missing information was the observational seeing. However, previous Grasiele aggregated that information on a table, which we will lead here and add to the current data:

In [31]:
data_cao = pd.read_csv("03_Cao_et_al._(2015)_Data.csv")
data_chen = pd.read_csv("03_Chen_et_al._(2019)_Data.csv")
previous = pd.read_csv("data/chen.csv")

In [32]:
data_cao["Cao_Seeing_atm"] = pd.NA
data_chen["Chen_Seeing_atm"] = pd.NA

In [33]:
data_cao.Cao_Survey.unique(), data_chen.Chen_Survey.unique()

(array(['BELLS', 'SLACS', 'LSD', 'SL2S'], dtype=object),
 array(['LSD', 'SL2S', 'SLACS', 'S4TM', 'BELLS', 'BELLS GALLERY'],
       dtype=object))

In [34]:
data_cao.loc[data_cao.query('Cao_Survey == "SLACS"').index, "Cao_Seeing_atm"] = 1.4
data_cao.loc[data_cao.query('Cao_Survey == "BELLS"').index, "Cao_Seeing_atm"] = 1.8
data_chen.loc[data_chen.query('Chen_Survey == "SLACS"').index, "Chen_Seeing_atm"] = 1.4
data_chen.loc[data_chen.query('Chen_Survey == "S4TM"').index, "Chen_Seeing_atm"] = 1.4
data_chen.loc[data_chen.query('Chen_Survey == "BELLS"').index, "Chen_Seeing_atm"] = 1.8
data_chen.loc[
    data_chen.query('Chen_Survey == "BELLS GALLERY"').index, "Chen_Seeing_atm"
] = 1.8

In [35]:
data_chen.query('Chen_Name=="SL2SJ020524−93023"')

,JNAME,Chen_Name,in_liu_2022,in_lastberu_cosmo_ground,Chen_z_L,Chen_z_S,Chen_velDisp,Chen_velDispErr,Chen_velDisp0,Chen_velDisp0Err,Chen_theta_E,Chen_theta_E_rad,Chen_theta_ap,Chen_theta_ap_rad,Chen_theta_eff,Chen_theta_eff_rad,Chen_slit,Chen_fiber_radius,Chen_Survey,Chen_Seeing_atm
5,NaN,SL2SJ020524−93023,False,False,0.557,1.33,276,37,287.334026,39.893352,0.76,0.000004,0.69,0.000003,0.75,0.000004,0.9 × 1.60,NaN,SL2S,<NA>


In [36]:
data_chen.loc[5, "Chen_Seeing_atm"] = 0.7

* LSD

In [37]:
previous.query('Survey == "LSD"')

,name,z_L,z_S,theta_E,theta_ap,velDisp,velDispErr,Survey,sigma_atm,theta_eff
0,MG2016+112,1.004,3.263,1.56,0.65,304,27,LSD,0.80,0.31
1,0047-281,0.485,3.595,1.34,0.41,219,12,LSD,0.70,0.82
2,CFRS03.1077,0.938,2.941,1.24,0.46,256,19,LSD,0.80,1.60
3,HST14176+5226,0.810,3.399,1.41,0.37,212,18,LSD,0.75,1.06
4,HST15433+5352,0.497,2.092,0.36,0.35,108,14,LSD,0.60,0.41


In [38]:
data_cao.query('Cao_Seeing_atm.isna() and Cao_Survey == "LSD"')

,JNAME,Cao_Name,in_cao_2017,in_lastberu_cosmo_ground,Cao_z_L,Cao_z_S,Cao_velDisp,Cao_velDispErr,Cao_velDisp0,Cao_velDisp0Err,Cao_theta_E,Cao_theta_E_rad,Cao_theta_ap,Cao_theta_ap_rad,Cao_theta_eff,Cao_theta_eff_rad,Cao_Survey,Cao_Seeing_atm
82,J004941.9-275226.0,Q0047-2808,True,False,0.485,3.595,229,15,246.483475,19.938945,1.34,0.000006,1.25,0.000006,0.82,0.000004,LSD,<NA>
83,J030230.7+000604.6,CFRS03-1077,True,False,0.938,2.941,251,19,258.503152,21.390205,1.24,0.000006,1.25,0.000006,1.60,0.000008,LSD,<NA>
84,J141736.1+522642.9,HST 14176,True,False,0.810,3.399,224,15,237.051044,18.644240,1.41,0.000007,1.25,0.000006,1.06,0.000005,LSD,<NA>
85,J154320.4+535152.0,HST 15433,False,False,0.497,2.092,116,10,130.700815,14.252374,0.36,0.000002,1.25,0.000006,0.41,0.000002,LSD,<NA>
86,J201918.1+112710.8,MG 2016,False,False,1.004,3.263,328,32,360.548948,40.561519,1.56,0.000008,0.65,0.000003,0.31,0.000002,LSD,<NA>


In [39]:
data_cao.loc[82, "Cao_Seeing_atm"] = 0.70
data_cao.loc[83, "Cao_Seeing_atm"] = 0.80
data_cao.loc[84, "Cao_Seeing_atm"] = 0.75
data_cao.loc[85, "Cao_Seeing_atm"] = 0.60
data_cao.loc[86, "Cao_Seeing_atm"] = 0.80

In [40]:
data_chen.query('Chen_Seeing_atm.isna() and Chen_Survey == "LSD"')

,JNAME,Chen_Name,in_liu_2022,in_lastberu_cosmo_ground,Chen_z_L,Chen_z_S,Chen_velDisp,Chen_velDispErr,Chen_velDisp0,Chen_velDisp0Err,Chen_theta_E,Chen_theta_E_rad,Chen_theta_ap,Chen_theta_ap_rad,Chen_theta_eff,Chen_theta_eff_rad,Chen_slit,Chen_fiber_radius,Chen_Survey,Chen_Seeing_atm
0,J201918.1+112710.8,MG2016+112,False,False,1.004,3.263,304,27,334.167318,35.089482,1.56,0.000008,0.65,0.000003,0.31,0.000002,1 × 1.25,NaN,LSD,<NA>
1,J004941.9-275226.0,0047−281,False,False,0.485,3.595,219,12,219.000000,13.680822,1.34,0.000006,0.41,0.000002,0.82,0.000004,0.4 × 1.25,NaN,LSD,<NA>
2,J030230.7+000604.6,CFRS03.1077,False,False,0.938,2.941,256,19,246.818690,20.266074,1.24,0.000006,0.46,0.000002,1.60,0.000008,0.5 × 1.25,NaN,LSD,<NA>
3,J141736.1+522642.9,HST14176+5226,False,False,0.810,3.399,212,18,207.030803,18.804068,1.41,0.000007,0.37,0.000002,1.06,0.000005,0.32 × 1.25,NaN,LSD,<NA>
4,J154320.4+535152.0,HSTT15433+5352,False,False,0.497,2.092,108,14,111.881039,15.016857,0.36,0.000002,0.35,0.000002,0.41,0.000002,0.3 × 1.25,NaN,LSD,<NA>


In [41]:
data_chen.loc[0, "Chen_Seeing_atm"] = 0.80
data_chen.loc[1, "Chen_Seeing_atm"] = 0.70
data_chen.loc[2, "Chen_Seeing_atm"] = 1.60
data_chen.loc[3, "Chen_Seeing_atm"] = 0.75
data_chen.loc[4, "Chen_Seeing_atm"] = 0.60

* SL2S

Because of missing data (from `Cao et al. (2015)`), we get the originals from "10.1088/0004-637X/777/2/98":

In [42]:
sl2s_data = pd.read_csv(
    "/home/renan/slcomp/02_Data/01_Processed_Data/Sonnenfeld_et_al._2013/dataset/SL2S_table_2_Sonnenfeld_1307.4759.csv"
)
sl2s_data.iloc[0:3]

,Name,obs. date,Instrument,slit,width,seeing,exp. time,zL,zS,velDisp,velDispErr,S/N,res.
0,SL2SJ020833-071414,11-29-2011,LRIS,1.0,1.62,1.0,900.0,0.428,NaN,295.0,27.0,17.0,150.0
1,SL2SJ021206-075528,01-28-2011,LRIS,0.7,1.62,0.6,2700.0,0.460,NaN,257.0,25.0,28.0,120.0
2,SL2SJ021247-055552,10-08-2010,XSHOOTER,0.9,1.60,0.7,2800.0,0.750,2.74,273.0,22.0,22.0,47.0


---

In [43]:
sl2s_data["Cao_Name"] = sl2s_data["Name"].str[4:9] + sl2s_data["Name"].str[11:16]

In [44]:
data_cao_sl2s = data_cao.query(
    'Cao_Seeing_atm.isna() and Cao_Survey == "SL2S"'
).reset_index(drop=True)
data_cao_sl2s.iloc[0:3]

,JNAME,Cao_Name,in_cao_2017,in_lastberu_cosmo_ground,Cao_z_L,Cao_z_S,Cao_velDisp,Cao_velDispErr,Cao_velDisp0,Cao_velDisp0Err,Cao_theta_E,Cao_theta_E_rad,Cao_theta_ap,Cao_theta_ap_rad,Cao_theta_eff,Cao_theta_eff_rad,Cao_Survey,Cao_Seeing_atm
0,J021247.0-055552.0,J0212-0555,True,False,0.750,2.74,273,22,280.098565,24.352231,1.27,0.000006,0.9,0.000004,1.22,0.000006,SL2S,<NA>
1,J021324.8-074354.9,J0213-0743,True,False,0.717,3.48,293,34,293.292413,35.153212,2.39,0.000012,1.0,0.000005,1.97,0.000010,SL2S,<NA>
2,J021411.1-040502.4,J0214-0405,True,False,0.609,1.88,287,47,296.678478,49.637910,1.41,0.000007,1.0,0.000005,1.21,0.000006,SL2S,<NA>


In [45]:
data_cao = data_cao.drop(
    index=data_cao.query('Cao_Seeing_atm.isna() and Cao_Survey == "SL2S"').index
)
data_cao_sl2s_fixed = (
    data_cao_sl2s.merge(sl2s_data[["Cao_Name", "seeing"]], on="Cao_Name")
    .drop(columns=["Cao_Seeing_atm"])
    .rename(columns={"seeing": "Cao_Seeing_atm"})
)

In [46]:
data_cao = pd.concat([data_cao, data_cao_sl2s_fixed]).reset_index(drop=True)
data_cao["Cao_Seeing_atm_rad"] = data_cao["Cao_Seeing_atm"] * u.arcsec.to(u.rad)

---

In [47]:
sl2s_data["Chen_Name"] = sl2s_data["Name"].str.replace("-", "−")

In [48]:
data_chen_sl2s = data_chen.query(
    'Chen_Seeing_atm.isna() and Chen_Survey == "SL2S"'
).reset_index(drop=True)
data_chen_sl2s.iloc[0:3]

,JNAME,Chen_Name,in_liu_2022,in_lastberu_cosmo_ground,Chen_z_L,Chen_z_S,Chen_velDisp,Chen_velDispErr,Chen_velDisp0,Chen_velDisp0Err,Chen_theta_E,Chen_theta_E_rad,Chen_theta_ap,Chen_theta_ap_rad,Chen_theta_eff,Chen_theta_eff_rad,Chen_slit,Chen_fiber_radius,Chen_Survey,Chen_Seeing_atm
0,J021247.0-055552.0,SL2SJ021247−055552,False,False,0.750,2.74,273,22,275.229460,23.693171,1.27,0.000006,0.69,0.000003,1.22,0.000006,0.9 × 1.60,NaN,SL2S,<NA>
1,J021324.8-074354.9,SL2SJ021325−074355,False,False,0.717,3.48,293,34,287.776199,34.588747,2.39,0.000012,0.75,0.000004,1.97,0.000010,1.0 × 1.68,NaN,SL2S,<NA>
2,J021411.1-040502.4,SL2SJ021411−040502,False,False,0.609,1.88,287,47,292.098569,48.698939,1.41,0.000007,0.79,0.000004,1.21,0.000006,1.0 × 1.88,NaN,SL2S,<NA>


In [49]:
data_chen = data_chen.drop(
    index=data_chen.query('Chen_Seeing_atm.isna() and Chen_Survey == "SL2S"').index
)
data_chen_sl2s_fixed = (
    data_chen_sl2s.merge(sl2s_data[["Chen_Name", "seeing"]], on="Chen_Name")
    .drop(columns=["Chen_Seeing_atm"])
    .rename(columns={"seeing": "Chen_Seeing_atm"})
)

In [50]:
data_chen = pd.concat([data_chen, data_chen_sl2s_fixed]).reset_index(drop=True)
data_chen["Chen_Seeing_atm_rad"] = data_chen["Chen_Seeing_atm"] * u.arcsec.to(u.rad)

In [51]:
data_cao.to_csv("03_Cao_et_al._(2015)_Data.csv", index=False)
data_chen.to_csv("03_Chen_et_al._(2019)_Data.csv", index=False)